# Estudio Comparativo: Navegación de Flappy Bird mediante LiDAR 🦇

En este experimento evolucionamos el clásico problema de control visual (basado en píxeles) hacia un enfoque de **robótica sensorial** utilizando telemetría LiDAR (Light Detection and Ranging). Utilizaremos el entorno `flappy-bird-gymnasium`.

Compararemos la eficacia de dos métodos de aprendizaje profundo por refuerzo operando sobre este nuevo espacio vectorial:
1. **DQN (Deep Q-Network)**: *Off-policy* con Replay Buffer.
2. **SARSA Semi-gradiente**: *On-policy* con aproximación de función lineal.

## Análisis de Contribución y Alineación Científica

La mayoría de implementaciones clásicas de Flappy Bird (como *Mnih et al., 2015*) utilizan **CNNs (Redes Convolucionales)** procesando imágenes crudas ($84 \times 84 \times 4 \approx 28,224$ inputs). Este enfoque es computacionalmente costoso y suele requerir un preprocesamiento complejo (escala de grises, recorte, apilado de frames).

Basándonos en la literatura reciente (*"Flappy Bird RL Performance using LiDAR", MDPI Sensors 2024*), proponemos las siguientes contribuciones de estudio:

1.  **Reducción de Dimensionalidad (The Curse of Dimensionality)**:
    Sustituimos la visión artificial por un **sensor LiDAR**. Esto reduce el espacio de estado de miles de píxeles a un vector de **180 distancias escalares** (rayos láser simulados). Esto permite usar redes **MLP (Perceptrones Multicapa)** simples para DQN y un aproximador lineal para SARSA, reduciendo drásticamente el número de parámetros y el tiempo de cómputo por paso.

2.  **Aceleración de Convergencia**:
    Al eliminar la necesidad de que la red "aprenda a ver" (extracción de características visuales de los bordes de las tuberías), el agente puede centrarse inmediatamente en aprender la **dinámica cinemática** y la relación distancia-acción.

3.  **Abstracción y Generalización**:
    Un agente basado en LiDAR es invariante a cambios estéticos (color de fondo, texturas, skins del pájaro) que confundirían a una CNN entrenada visualmente. Esto se acerca más a la navegación de robots móviles reales.

### Referencias
- Entorno: [flappy-bird-gymnasium](https://github.com/markub3327/flappy-bird-gymnasium)
- Artículo base: [MDPI Sensors 2024](https://www.mdpi.com/1424-8220/24/6/1905)

In [1]:
# Instalación del entorno específico con soporte LiDAR
# Descomentar si no está instalado:
# !pip install flappy-bird-gymnasium gymnasium torch matplotlib imageio

import gymnasium as gym
import flappy_bird_gymnasium
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import random
import matplotlib.pyplot as plt
from collections import deque
import imageio
import base64
from IPython.display import HTML, display

# Configuración de Semilla para Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilizando dispositivo de cómputo: {device}")

Utilizando dispositivo de cómputo: cuda


## 1. Configuración del Entorno LiDAR

A diferencia del enfoque visual, aquí no necesitamos `FrameStack` (memoria de inercia visual) ni conversión a escala de grises. El entorno `FlappyBird-v0` con la opción `use_lidar=True` nos devuelve directamente un **vector 1D** que representa las distancias a los obstáculos (suelo, techo, tuberías) en un abanico radial alrededor del pájaro.

In [2]:
def make_env(render_mode=None):
    # use_lidar=True activa el sensor de distancia en lugar de la imagen RGB
    env = gym.make("FlappyBird-v0", render_mode=render_mode, use_lidar=True)
    return env

# Inspección del Espacio de Observación
temp_env = make_env()
obs_dim = temp_env.observation_space.shape[0]
action_dim = temp_env.action_space.n
print(f"Dimensión del Sensor LiDAR: {obs_dim} rayos (Inputs)")
print(f"Espacio de Acción: {action_dim} (0: No aletear, 1: Aletear)")
temp_env.close()

Dimensión del Sensor LiDAR: 180 rayos (Inputs)
Espacio de Acción: 2 (0: No aletear, 1: Aletear)


## Justificación Teórica: ¿Por qué NO usar métodos Tabulares?

Al activar el sensor LiDAR, transformamos la observación en un vector de **180 valores continuos** (números flotantes que representan distancias).

### Análisis de la Maldición de la Dimensionalidad
Un método tabular (como Q-Learning estándar) requeriría una matriz $Q(s, a)$. Incluso si simplificamos radicalmente el problema discretizando los valores:

1.  Supongamos que discretizamos cada uno de los 180 láseres en solo **2 estados** (cerca/lejos).
    $$ |S| = 2^{180} \approx 1.53 \times 10^{54} \text{ estados combinatorios} $$
    *(Esta magnitud excede por mucho la memoria de cualquier computador existente)*

2.  En realidad, las distancias son valores continuos (`float32`). El espacio de estados es **infinito no numerable** ($\mathbb{R}^{180}$).

### Conclusión
Es **imposible** almacenar o visitar todos los estados en una tabla. Necesitamos obligatoriamente **Aproximación de Funciones** que puedan:
1.  Manejar entradas continuas sin sufrir una explosión combinatoria.
2.  **Generalizar**: Entender que una distancia de `0.51` es cinemáticamente casi idéntica a `0.52`. Un método tabular trataría estos valores como casillas totalmente independientes sin compartir aprendizaje entre ellas.

Por esta razón fundamental, el uso de aproximación de funciones (como **Deep Q-Learning** o **SARSA Semi-gradiente**) no es opcional, sino necesario.

## 2. Arquitectura de Red y Función de Valor

Abandonamos las costosas CNNs. Para **DQN** usaremos una red *feedforward* (totalmente conectada) simple (MLP) utilizando PyTorch. 

Para **SARSA Semi-gradiente** no usaremos PyTorch, sino que aplicaremos la definición matemática pura del algoritmo mediante Numpy, estimando la función de valor paramétrica explícitamente como el producto escalar de un vector de pesos y el estado: $\hat{q}(s,a,w) = w^\top x(s)$. Esto nos permite programar la regla de actualización algorítmica y el cálculo de gradientes directamente según el pseudocódigo.

In [3]:
class LidarNetwork(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=64, num_layers=3):
        super(LidarNetwork, self).__init__()
        # Usamos num_layers para determinar la profundidad dinámicamente
        self.layers = nn.ModuleList()
        
        if num_layers == 1:
            self.layers.append(nn.Linear(input_dim, output_dim))
        else:
            self.layers.append(nn.Linear(input_dim, hidden_dim))
            for _ in range(num_layers - 2):
                self.layers.append(nn.Linear(hidden_dim, hidden_dim))
            self.layers.append(nn.Linear(hidden_dim, output_dim))
            
    def forward(self, x):
        for i in range(len(self.layers) - 1):
            x = F.relu(self.layers[i](x))
        return self.layers[-1](x)

## 3. Implementación de Agentes (DQN vs SARSA Semi-gradiente)

Definimos dos especializaciones para estudiar sus diferencias basándonos en los algoritmos documentados:

- **DQN (Deep Q-learning with Experience Replay)**: Implementado de acuerdo al algoritmo propuesto originalmente en NIPS 2013. Mantiene una *Replay Memory* limitadad de tamaño para romper la correlación temporal y realiza un *gradient descent step* calculando la pérdida del error cuadrático sobre su propia red neuronal. No utiliza *Target Network* en esta versión formativa.
- **SARSA Semi-gradiente (On-Policy)**: Implementado estrictamente siguiendo el pseudocódigo clásico iterativo. Calcula manualmente la aproximación de valor usando el producto punto $\hat{q}(S,A,w) = w^\top x(s)$ y realiza la actualización paso a paso mediante $w \leftarrow w + \alpha [R + \gamma \hat{q}(S',A',w) - \hat{q}(S,A,w)] \nabla \hat{q}(S,A,w)$.

In [4]:
class ReplayBuffer:
    def __init__(self, capacity):
        # Initialize replay memory D to capacity N
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        # Store transition (phi_t, a_t, r_t, phi_t+1) in D
        # Notamos que "done" se guarda para saber el comportamiento de phj_j+1
        self.buffer.append((state, action, reward, next_state, done))
        
    def sample(self, batch_size):
        # Sample random minibatch of transitions from D
        batch = random.sample(self.buffer, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (np.array(s), np.array(a), np.array(r, dtype=np.float32), 
                np.array(ns), np.array(d, dtype=np.float32))
    
    def __len__(self):
        return len(self.buffer)

# --- Implementación Clásica de Deep Q-learning with Experience Replay (2013) ---
# Nota: La versión original de Mnih et al. 2013 no tenía una "Target Network" (red objetivo separada)
# como lo hicieron en su Nature Paper de 2015. Utilizaban los pesos θ actuales para y_j.
class DQNAgent:
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99, epsilon_start=1.0, epsilon_decay=0.995, hidden_dim=64, num_layers=3):
        # Initialize action-value function Q with random weights
        self.q_net = LidarNetwork(state_dim, action_dim, hidden_dim=hidden_dim, num_layers=num_layers).to(device)
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        
        # Initialize replay memory D to capacity N (e.g. 10000)
        self.memory = ReplayBuffer(10000)
        self.batch_size = 64
        
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_min = 0.01
        self.epsilon_decay = epsilon_decay
        self.action_dim = action_dim
        
    def act(self, state, evaluate=False):
        # With probability epislon select a random action a_t
        if not evaluate and random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        
        # otherwise select a_t = max_a Q(phi(st), a; theta)
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            q_values = self.q_net(state_t)
        return torch.argmax(q_values).item()
        
    def update(self, state, action, reward, next_state, done, next_action=None):
        # Store transition in D
        self.memory.push(state, action, reward, next_state, done)
        
        # Si el replay buffer no tiene suficientes muestras, no entrenar aún
        if len(self.memory) < self.batch_size: return
        
        # Sample random minibatch of transitions
        s, a, r, ns, d = self.memory.sample(self.batch_size)
        
        states = torch.FloatTensor(s).to(device)
        actions = torch.LongTensor(a).unsqueeze(1).to(device)
        rewards = torch.FloatTensor(r).unsqueeze(1).to(device)
        next_states = torch.FloatTensor(ns).to(device)
        dones = torch.FloatTensor(d).unsqueeze(1).to(device)
        
        # Perform a gradient descent step on (y_j - Q(phi_j, a_j; theta))^2
        curr_Q = self.q_net(states).gather(1, actions)
        
        # Cálculo de y_j
        with torch.no_grad():
            # max_a_prime Q(phi_j+1, a_prime; theta)
            # Como es el DQN original de 2013, se usa q_net (theta) directamente, no target_net.
            max_next_Q = self.q_net(next_states).max(1)[0].unsqueeze(1)
            
            # y_j = r_j if terminal OR r_j + gamma * max_a' Q(next_state, a'; theta) if non-terminal
            # 1 - dones evalúa a 0 si es terminal (dejando sumando solo la recompensa) y 1 si no lo es.
            y_j = rewards + (self.gamma * max_next_Q * (1 - dones))
            
        loss = F.mse_loss(curr_Q, y_j)
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        if self.epsilon > self.epsilon_min: self.epsilon *= self.epsilon_decay

# --- Implementación Mejorada de SARSA Semi-gradiente Episódico ---
# Corrección: Se añade término de sesgo (bias) explícito al modelo lineal.
class SARSAAgent:
    def __init__(self, state_dim, action_dim, lr=1e-4, gamma=0.99, epsilon_start=1.0, epsilon_decay=0.995):
        # Initialize value-function weights w \in R^d arbitrarily (e.g., w = 0)
        # IMPORTANTE: Añadimos una dimensión extra para el BIAS (sesgo)
        # Esto permite que la aproximación lineal tenga un término independiente (ordenada en el origen)
        # sin el cual estaría forzada a pasar por el origen (0,0), limitando severamente el aprendizaje.
        self.w = np.zeros((action_dim, state_dim + 1))
        
        # Algorithm parameters: step size alpha > 0 (lr), small epsilon > 0
        self.alpha = lr  
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_min = 0.01
        self.epsilon_decay = epsilon_decay
        self.action_dim = action_dim
        
    def get_features(self, state):
        # Añade un 1.0 al final del vector de estado para actuar como bias
        return np.append(state, 1.0)
        
    def q_hat(self, state, action):
        # A differentiable action-value function parameterization q_hat(s,a,w)
        # q_hat = w^T * x(s) (incluyendo el bias en w y x(s))
        features = self.get_features(state)
        return np.dot(self.w[action], features)
        
    def act(self, state, evaluate=False):
        # Choose A as a function of q_hat(S, ·, w) (e.g., e-greedy)
        if not evaluate and random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        
        q_values = [self.q_hat(state, a) for a in range(self.action_dim)]
        return np.argmax(q_values)
        
    def update(self, state, action, reward, next_state, done, next_action):
        # Features actuales
        features = self.get_features(state)
        
        # Evaluamos el valor actual q_hat(S, A, w)
        q_current = np.dot(self.w[action], features)
        
        # El gradiente ∇q_hat(S, A, w) respecto a los pesos `w` de la acción tomada 
        # es exactamente el vector de características (incluyendo bias)
        nabla_q = features
        
        if done:
            # If S' is terminal: 
            # w <- w + a * [R - q_hat(S,A,w)] * ∇q_hat(S,A,w)
            error = reward - q_current
        else:
            # w <- w + a * [R + gamma * q_hat(S',A',w) - q_hat(S,A,w)] * ∇q_hat(S,A,w)
            q_next = self.q_hat(next_state, next_action)
            error = reward + self.gamma * q_next - q_current
            
        # Aplicamos la actualización del semi-gradiente para la acción A
        # Clipping del gradiente opcional pero recomendado para estabilidad en RL con floats
        # self.w[action] += np.clip(self.alpha * error * nabla_q, -1, 1) 
        self.w[action] += self.alpha * error * nabla_q
        
        # Decaer exploración episódica (procedimiento estándar para e-greedy a lo largo de los pasos/episodios)
        if self.epsilon > self.epsilon_min: 
            self.epsilon *= self.epsilon_decay

## 4. Experimentación: Impacto del Factor de Descuento ($\gamma$)

Vamos a evaluar cómo el horizonte temporal (el factor de descuento $\gamma$) afecta al aprendizaje probando los 2 algoritmos con esta variabilidad:
- **$\gamma = 0.1$** (Bajo): El agente es muy miope, solo se preocupa por la recompensa inmediata.
- **$\gamma = 0.7$** (Medio): Equilibrio entre recompensas a corto y medio plazo.
- **$\gamma = 0.99$** (Alto): El agente evalúa las consecuencias a largo plazo, crucial para prever colisiones futuras con tuberías en Flappy Bird.

Para cada algoritmo mediremos tres métricas fundamentales episodio a episodio:
1. **Puntuación Media**: Recompensa total obtenida durante el episodio (rendimiento puro).
2. **Tiempo de Vuelo Promedio**: Cantidad de pasos que el pájaro logra sobrevivir. Con un $\gamma$ bajo que ignore los castigos futuros, ¿intentará siquiera volar o chocará enseguida?
3. **Estabilidad de la Función Q**: Valor máximo predicho para $Q$ en cada episodio (las esperanzas matemáticas del agente). Un $\gamma$ alto tiende a acumular valores de $Q$ más altos y refleja qué tanto predice el éxito futuro.

In [ ]:
def train_agent(agent_class, name, episodes=100, gamma=0.99, hidden_dim=64, num_layers=3):
    env = make_env()
    if agent_class == DQNAgent:
        agent = agent_class(obs_dim, action_dim, lr=1e-4, gamma=gamma, hidden_dim=hidden_dim, num_layers=num_layers)
    else:
        agent = agent_class(obs_dim, action_dim, lr=1e-4, gamma=gamma)
    
    scores = []
    flight_times = []
    max_q_values = []
    
    print(f"--- Iniciando: {name} | Gamma: {gamma} | Nodos: {hidden_dim} | Capas: {num_layers} ---")
    
    for i in range(episodes):
        state, _ = env.reset(seed=SEED + i)
        done = False
        score = 0
        steps = 0
        ep_max_qs = []
        
        # Para SARSA necesitamos elegir la primera acción antes del bucle
        action = agent.act(state)
        
        while not done:
            steps += 1
            
            # Registrar el valor máximo de Q en el estado actual para monitorizar
            if isinstance(agent, DQNAgent):
                action = agent.act(state)
                state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
                with torch.no_grad():
                    q_vals = agent.q_net(state_t)
                    ep_max_qs.append(torch.max(q_vals).item())
            else:
                q_vals = [agent.q_hat(state, a) for a in range(agent.action_dim)]
                ep_max_qs.append(np.max(q_vals))
            
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            score += reward
            
            if isinstance(agent, SARSAAgent):
                # On-policy: Elegir siguiente acción a' según política actual
                next_action = agent.act(next_state)
                agent.update(state, action, reward, next_state, done, next_action)
                state = next_state
                action = next_action
            else:
                # Off-policy (DQN): Solo guardamos la experiencia
                agent.update(state, action, reward, next_state, done)
                state = next_state
                
        scores.append(score)
        flight_times.append(steps)
        # Promedio del max Q a lo largo los steps del episodio
        max_q_values.append(np.mean(ep_max_qs) if ep_max_qs else 0)
        
        if (i+1) % 100 == 0:
            avg_score = np.mean(scores[-50:])
            print(f"Episodio {i+1}: Media Score = {avg_score:.2f} | Vuelo = {np.mean(flight_times[-50:]):.1f} steps | Q-max = {np.mean(max_q_values[-50:]):.2f}")
            
    env.close()
    return agent, scores, flight_times, max_q_values

# ----------------- EJECUCIÓN EXPERIMENTO 1: Gamma -----------------
gammas = [0.1, 0.7, 0.99]
episodes = 5000 

results_dqn = {}
results_sarsa = {}

print("=== INICIANDO EXPERIMENTO 1: FACTOR DE DESCUENTO GAMMA ===")
for g in gammas:
    dqn_ag, dqn_s, dqn_f, dqn_q = train_agent(DQNAgent, "DQN", episodes=episodes, gamma=g)
    results_dqn[g] = {'scores': dqn_s, 'flights': dqn_f, 'qmax': dqn_q}
    
    sarsa_ag, sarsa_s, sarsa_f, sarsa_q = train_agent(SARSAAgent, "SARSA SG", episodes=episodes, gamma=g)
    results_sarsa[g] = {'scores': sarsa_s, 'flights': sarsa_f, 'qmax': sarsa_q}

=== INICIANDO EXPERIMENTO 1: FACTOR DE DESCUENTO GAMMA ===
--- Iniciando: DQN | Gamma: 0.1 | Nodos: 64 | Capas: 3 ---


c:\Users\AntonioLuisTorres\.conda\envs\eml-flappy\Lib\site-packages\gymnasium\utils\passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
c:\Users\AntonioLuisTorres\.conda\envs\eml-flappy\Lib\site-packages\gymnasium\utils\passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


Episodio 100: Media Score = -1.33 | Vuelo = 50.8 steps | Q-max = -0.02
Episodio 200: Media Score = -1.59 | Vuelo = 50.5 steps | Q-max = -0.03
Episodio 300: Media Score = -1.70 | Vuelo = 50.1 steps | Q-max = -0.03
Episodio 400: Media Score = -1.48 | Vuelo = 51.9 steps | Q-max = -0.03
Episodio 500: Media Score = -1.09 | Vuelo = 50.6 steps | Q-max = -0.02
Episodio 600: Media Score = -2.45 | Vuelo = 50.2 steps | Q-max = -0.04
Episodio 700: Media Score = -1.80 | Vuelo = 50.0 steps | Q-max = -0.04
Episodio 800: Media Score = -2.18 | Vuelo = 50.6 steps | Q-max = -0.04
Episodio 900: Media Score = -2.94 | Vuelo = 50.0 steps | Q-max = -0.05


KeyboardInterrupt: 

## 5. Visualización de los Resultados

Analizamos en profundidad las 3 variables registradas para cada valor de gamma en ambos esquemas de aprendizaje (Off-Policy Neuronal vs On-Policy Lineal).

**Hipótesis esperada:**
- Con **Gamma 0.1** los Q-values serán minúsculos y su "Tiempo de Vuelo" será tan corto que indicará un choque frontal temprano contra los obstáculos por no estar previendo las penalizaciones.
- Con **Gamma 0.99** crecerá la estabilidad de estimación de la función de Valor (líneas de Q-max ascendentes y ricas en información), correlacionando fuertemente con un mayor de Tiempo de Vuelo.

In [ ]:
def moving_average(data, window_size=25):
    return np.convolve(data, np.ones(window_size)/window_size, mode='valid')

fig, axs = plt.subplots(3, 2, figsize=(16, 18))
fig.suptitle('Exp. paramétrico: Evolución de DQN y SARSA variando el Horizonte Temporal ($\gamma$)', fontsize=16)

colors = {0.1: 'red', 0.7: 'green', 0.99: 'blue'}

# ===== Parte DQN =====
for g in gammas:
    axs[0, 0].plot(moving_average(results_dqn[g]['scores']), label=f'$\gamma={g}$', color=colors[g], alpha=0.7)
    axs[1, 0].plot(moving_average(results_dqn[g]['flights']), label=f'$\gamma={g}$', color=colors[g], alpha=0.7)
    axs[2, 0].plot(moving_average(results_dqn[g]['qmax']), label=f'$\gamma={g}$', color=colors[g], alpha=0.7)

axs[0, 0].set_title('DQN: Puntuación Media (Reward)')
axs[0, 0].set_ylabel('Scores Suavizados')
axs[1, 0].set_title('DQN: Tiempo de Vuelo')
axs[1, 0].set_ylabel('Pasos por Episodio (Supervivencia)')
axs[2, 0].set_title('DQN: Estabilidad de Q')
axs[2, 0].set_ylabel('Promedio del Max Q-Value')

# ===== Parte SARSA ======
for g in gammas:
    axs[0, 1].plot(moving_average(results_sarsa[g]['scores']), label=f'$\gamma={g}$', color=colors[g], alpha=0.7)
    axs[1, 1].plot(moving_average(results_sarsa[g]['flights']), label=f'$\gamma={g}$', color=colors[g], alpha=0.7)
    axs[2, 1].plot(moving_average(results_sarsa[g]['qmax']), label=f'$\gamma={g}$', color=colors[g], alpha=0.7)

axs[0, 1].set_title('SARSA Semi-Grad: Puntuación Media')
axs[1, 1].set_title('SARSA Semi-Grad: Tiempo de Vuelo')
axs[2, 1].set_title('SARSA Semi-Grad: Estabilidad de Q')

for ax in axs.flat:
    ax.set_xlabel('Episodios')
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.show()

## 6. Experimentación: Densidad de Información del Sensor LiDAR

Vamos a realizar un segundo experimento comparando cómo afecta la **densidad de la observación** al rendimiento de ambos algoritmos. La observación base consiste en 180 rayos distribuidos uniformemente. Probaremos las siguientes configuraciones reduciendo el vector de entrada:

- **Configuración A (Minimalista)**: 10 rayos (solo seleccionando índices representativos distribuidos: `0, 18, 36...`).
- **Configuración B (Reducida)**: 90 rayos (solo índices pares `0, 2, 4...`).
- **Configuración C (Completa)**: 180 rayos (vector íntegro).

Este experimento nos dirá si tanta información (180 rayos) es realmente necesaria para la toma de decisiones o si, de hecho, el exceso de variables ralentiza el aprendizaje.

In [ ]:
class MaskedEnvWrapper(gym.ObservationWrapper):
    """Wrapper para reducir el número de rayos procesados por el agente"""
    def __init__(self, env, num_rays):
        super().__init__(env)
        self.num_rays = num_rays
        # Calculamos los índices a mantener equiespaciados
        self.keep_indices = np.linspace(0, 179, num_rays, dtype=int)
        
        # Redefinimos el espacio de observación
        old_space = env.observation_space
        self.observation_space = gym.spaces.Box(
            low=old_space.low[0], 
            high=old_space.high[0], 
            shape=(num_rays,), 
            dtype=np.float32
        )

    def observation(self, obs):
        return obs[self.keep_indices]

def train_agent_masked(agent_class, name, episodes=100, num_rays=180, gamma=0.99):
    env = make_env()
    if num_rays < 180:
        env = MaskedEnvWrapper(env, num_rays)
    
    # Creamos el agente con la dimensión de entrada reducida
    if agent_class == DQNAgent:
        agent = agent_class(num_rays, action_dim, lr=1e-4, gamma=gamma, hidden_dim=64, num_layers=3)
    else:
        agent = agent_class(num_rays, action_dim, lr=1e-4, gamma=gamma)
    
    scores = []
    flight_times = []
    
    print(f"--- Iniciando: {name} | Rayos: {num_rays} ---")
    
    for i in range(episodes):
        state, _ = env.reset(seed=SEED + i)
        done = False
        score = 0
        steps = 0
        
        action = agent.act(state)
        
        while not done:
            steps += 1
            if isinstance(agent, DQNAgent):
                action = agent.act(state)
            
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            score += reward
            
            if isinstance(agent, SARSAAgent):
                next_action = agent.act(next_state)
                agent.update(state, action, reward, next_state, done, next_action)
                state = next_state
                action = next_action
            else:
                agent.update(state, action, reward, next_state, done)
                state = next_state
                
        scores.append(score)
        flight_times.append(steps)
        
        if (i+1) % 50 == 0:
            avg_score = np.mean(scores[-50:])
            print(f"Episodio {i+1}: Media Score = {avg_score:.2f} | Vuelo = {np.mean(flight_times[-50:]):.1f} steps")
            
    env.close()
    return scores, flight_times

# ----------------- EJECUCIÓN EXPERIMENTO 2: DENSIDAD LIDAR -----------------
ray_configs = [10, 90, 180]
episodes_rays = 200 

results_rays_dqn = {}
results_rays_sarsa = {}

print("=== INICIANDO EXPERIMENTO 2: DENSIDAD DEL SENSOR LIDAR ===")
for rays in ray_configs:
    scores, flights = train_agent_masked(DQNAgent, "DQN", episodes=episodes_rays, num_rays=rays)
    results_rays_dqn[rays] = {'scores': scores, 'flights': flights}
    
    scores, flights = train_agent_masked(SARSAAgent, "SARSA SG", episodes=episodes_rays, num_rays=rays)
    results_rays_sarsa[rays] = {'scores': scores, 'flights': flights}

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Impacto de la Densidad del Lidar (Número de Rayos)', fontsize=16)

ray_colors = {10: '#ff0000', 90: '#ed9b00', 180: '#0000ff'}

# Gráficas DQN
for r in ray_configs:
    axs[0, 0].plot(moving_average(results_rays_dqn[r]['scores']), label=f'{r} Rayos', color=ray_colors[r], alpha=0.8)
    axs[1, 0].plot(moving_average(results_rays_dqn[r]['flights']), label=f'{r} Rayos', color=ray_colors[r], alpha=0.8)

axs[0, 0].set_title('DQN: Puntuación Media Suavizada')
axs[0, 0].set_ylabel('Scores')
axs[0, 0].legend()
axs[0, 0].grid(True, alpha=0.3)

axs[1, 0].set_title('DQN: Tiempo de Supervivencia')
axs[1, 0].set_ylabel('Pasos por Episodio')
axs[1, 0].set_xlabel('Episodios')
axs[1, 0].legend()
axs[1, 0].grid(True, alpha=0.3)

# Gráficas SARSA
for r in ray_configs:
    axs[0, 1].plot(moving_average(results_rays_sarsa[r]['scores']), label=f'{r} Rayos', color=ray_colors[r], alpha=0.8)
    axs[1, 1].plot(moving_average(results_rays_sarsa[r]['flights']), label=f'{r} Rayos', color=ray_colors[r], alpha=0.8)

axs[0, 1].set_title('SARSA SG: Puntuación Media Suavizada')
axs[0, 1].legend()
axs[0, 1].grid(True, alpha=0.3)

axs[1, 1].set_title('SARSA SG: Tiempo de Supervivencia')
axs[1, 1].set_xlabel('Episodios')
axs[1, 1].legend()
axs[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Experimentación: Impacto de la Arquitectura de Red en DQN

Ahora que hemos comprobado el efecto del Factor de Descuento sobre ambos algoritmos, nos limitaremos a **DQN** ($\gamma=0.99$) para evaluar un hiperparámetro exclusivo del Aprendizaje Profundo: la topología de la Red Neuronal (DQN Network).

El espacio físico cuenta con $180$ distancias continuas. Queremos probar si existe **overfitting** (al usar redes demasiado amplias y profundas para 180 inputs) o un posible **underfitting** (redes que no capturen la política). Probaremos las siguientes arquitecturas de red densa (indicadas en *capas* $\times$ *nodos por capa*):

- **1x64**: $(180 \rightarrow 64 \rightarrow 2)$
- **3x64**: $(180 \rightarrow 64 \rightarrow 64 \rightarrow 64 \rightarrow 2)$
- **5x64**: $(180 \rightarrow 64 \rightarrow \text{...} \rightarrow 64 \rightarrow 2)$
- **1x512**: $(180 \rightarrow 512 \rightarrow 2)$ - *Wide Network*
- **3x128**: $(180 \rightarrow 128 \rightarrow 128 \rightarrow 128 \rightarrow 2)$ - *Estructura "Diamond"*
- **5x256**: $(180 \rightarrow 256 \rightarrow \text{...} \rightarrow 256 \rightarrow 2)$ - *Deep & Wide (Riesgo de colapso de gradiente en pocos episodios)*

In [ ]:
# ----------------- EJECUCIÓN EXPERIMENTO 3: Topología DQN -----------------
architectures = [
    (1, 64),
    (3, 64),
    (5, 64),
    (1, 512),
    (3, 256),
    (5, 256)
]

episodes_arch = 2000 
results_arch = {}

print("=== INICIANDO EXPERIMENTO 3: ESTUDIO DE TOPOLOGÍAS DQN ===")
for layers, hidden_dim in architectures:
    name = f"DQN_{layers}x{hidden_dim}"
    _, scores, flights, _ = train_agent(DQNAgent, name, episodes=episodes_arch, gamma=0.99, hidden_dim=hidden_dim, num_layers=layers)
    results_arch[name] = {'scores': scores, 'flights': flights}

### 7.1. Visualización del Impacto Arquitectónico en DQN

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Impacto de Capas Ocultas y Nodos en el Aprendizaje (DQN)', fontsize=16)

# Mapa de colores para las arquitecturas
arch_colors = {
    'DQN_1x64': '#1f77b4', 'DQN_3x64': '#ff7f0e', 'DQN_5x64': '#2ca02c',
    'DQN_1x512': '#d62728', 'DQN_3x256': '#9467bd', 'DQN_5x256': '#8c564b'
}

for name in results_arch:
    axs[0].plot(moving_average(results_arch[name]['scores']), label=name, color=arch_colors[name], alpha=0.8)
    axs[1].plot(moving_average(results_arch[name]['flights']), label=name, color=arch_colors[name], alpha=0.8)

axs[0].set_title('Puntuación Media Suavizada')
axs[0].set_ylabel('Scores')
axs[0].set_xlabel('Episodios')
axs[0].legend()
axs[0].grid(True, alpha=0.3)

axs[1].set_title('Tiempo de Supervivencia (Steps)')
axs[1].set_ylabel('Pasos por Episodio')
axs[1].set_xlabel('Episodios')
axs[1].legend()
axs[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Entrenamiento Final de la Mejor Configuración y Visualización (GIF)

Basándonos en los resultados de los experimentos anteriores, la mejor combinación observada ha resultado ser:
- **Agente**: DQN 
- **Gamma**: 0.99 (Planificación a largo plazo asegurada)
- **Densidad LiDAR**: 90 rayos (Información reducida óptima, filtrando detalles innecesarios)
- **Arquitectura de Red**: 3 capas densas de 64 nodos (Suficiente capacidad sin sobreajuste rápido).

Dado que ya tenemos la configuración, vamos a realizar un entrenamiento intensivo de esta configuración optimizada durante **2000 episodios** y guardaremos y visualizaremos una partida del fruto final como un GIF animado.

In [ ]:
def train_best_agent(agent_class, episodes=2000, gamma=0.99, hidden_dim=64, num_layers=3, num_rays=90):
    env = make_env()
    if num_rays < 180:
        env = MaskedEnvWrapper(env, num_rays)
    
    # El espacio de acción de FlappyBird es 2
    agent = agent_class(num_rays, env.action_space.n, lr=1e-4, gamma=gamma, hidden_dim=hidden_dim, num_layers=num_layers)
    
    scores, flight_times = [], []
    for i in range(episodes):
        state, _ = env.reset(seed=SEED + i)
        done = False
        score = steps = 0
        
        while not done:
            steps += 1
            # Importante: Como es DQN_Best_Config, sacamos la acción actual
            action = agent.act(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            score += reward
            
            agent.update(state, action, reward, next_state, done)
            state = next_state
                
        scores.append(score)
        flight_times.append(steps)
        if (i+1) % 100 == 0:
            print(f"Episodio {i+1}: Media Score = {np.mean(scores[-50:]):.2f} | Vuelo = {np.mean(flight_times[-50:]):.1f} steps")
            
    env.close()
    return agent, scores, flight_times

def save_agent_gif(agent, filename="flappy_bird_lidar_best.gif", num_rays=90):
    env = make_env(render_mode="rgb_array") 
    if num_rays < 180:
        env = MaskedEnvWrapper(env, num_rays)
        
    state, _ = env.reset(seed=10)
    frames = []
    done = False
    
    while not done:
        frames.append(env.render())
        action = agent.act(state, evaluate=True) 
        state, _, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        if len(frames) > 1000: break # Límite de seguridad
            
    env.close()
    
    if len(frames) > 0:
        imageio.mimsave(filename, frames, fps=30)
        return filename
    return None

print("=== INICIANDO ENTRENAMIENTO FINAL (2000 Episodios) ===")
# Llamamos a nuestra configuración ganadora específicamente con 90 rayos
best_agent, s, f = train_best_agent(
    agent_class=DQNAgent, 
    episodes=2000, 
    gamma=0.99, 
    hidden_dim=64, 
    num_layers=3,
    num_rays=90
)

print("\nGenerando GIF del mejor agente entrenado con 90 rayos...")
gif_path = save_agent_gif(best_agent, num_rays=90)

if gif_path:
    with open(gif_path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(f'<h3>Comportamiento Aprendido vs 2000 Episodios (90 Rayos)</h3><img src="data:image/gif;base64,{b64}" width="400"/>'))
    
    # Opcional: mostrar una pequeña gráfica del entrenamiento de esta config final
    plt.figure(figsize=(10,4))
    plt.plot(moving_average(s, 50), label="Puntuación Media")
    plt.title("Rendimiento de la Configuración Final Óptima (90 Rayos)")
    plt.xlabel("Episodios")
    plt.ylabel("Score Suavizado")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("No se pudo generar el GIF.")